# NBA play-by-play in Python

Connects to the MotherDuck `nba` database. Run SQL with `q(...)` to get a pandas DataFrame, then use pandas or matplotlib as usual.

Needs `MOTHERDUCK_TOKEN` set in your environment (restart VS Code after setting it).

In [ ]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt

con = duckdb.connect("md:nba")

def q(sql, *params):
    """Run SQL against MotherDuck and return a pandas DataFrame."""
    return con.execute(sql, list(params)).df()

q("SHOW TABLES")

## Look up a player

In [ ]:
player = q("SELECT * FROM dim_players WHERE full_name ILIKE ?", "%gilgeous%")
player_id = int(player.player_id.iloc[0])
player[["player_id", "full_name", "team", "position", "height", "jersey"]]

## Points per game from the play-by-play

In [ ]:
games = q("""
    SELECT g.game_date, g.away_team || ' @ ' || g.home_team AS matchup,
           sum(p.shot_value) FILTER (p.shot_result = 'Made') AS pts,
           count(*) FILTER (p.is_field_goal) AS fga
    FROM pbp p JOIN games g USING (game_id)
    WHERE p.person_id = ?
    GROUP BY ALL ORDER BY g.game_date
""", player_id)
print(f"{len(games)} games, {games.pts.mean():.1f} PPG")
games.tail(10)

## Shot chart

`x_legacy` / `y_legacy` are in tenths of a foot, measured from the basket.

In [ ]:
shots = q("""
    SELECT x_legacy, y_legacy, shot_result
    FROM pbp WHERE person_id = ? AND is_field_goal AND x_legacy IS NOT NULL
""", player_id)

fig, ax = plt.subplots(figsize=(6, 5.6))
styles = {"Made": dict(color="#2a78d6", marker="o"), "Missed": dict(color="#eb6834", marker="x")}
for result, s in shots.groupby("shot_result"):
    ax.scatter(s.x_legacy, s.y_legacy, s=18, alpha=0.6, linewidths=1.5,
               label=f"{result} ({len(s)})", **styles[result])

# Basket, 3-point arc and baseline for reference
ax.add_patch(plt.Circle((0, 0), 7.5, fill=False, color="#888", lw=1))
ax.add_patch(plt.matplotlib.patches.Arc((0, 0), 475, 475, theta1=22, theta2=158, color="#888", lw=1))
ax.plot([-220, -220], [-47.5, 92.5], color="#888", lw=1)
ax.plot([220, 220], [-47.5, 92.5], color="#888", lw=1)
ax.axhline(-47.5, color="#888", lw=1)

ax.set_xlim(-250, 250); ax.set_ylim(-50, 400); ax.set_aspect("equal")
ax.set_xticks([]); ax.set_yticks([])
for side in ax.spines.values():
    side.set_visible(False)
ax.set_title(f"{player.full_name.iloc[0]}: field-goal attempts", loc="left")
ax.legend(loc="upper right", frameon=False)
plt.show()

## Who guarded him most (matchups)

In [ ]:
q("""
    SELECT def_player,
           count(DISTINCT game_id)              AS games,
           round(sum(matchup_seconds) / 60, 1)  AS minutes,
           sum(player_points)                   AS pts,
           sum(fgm) || '/' || sum(fga)          AS fg,
           round(sum(fgm) / nullif(sum(fga), 0), 3) AS fg_pct
    FROM matchups WHERE off_person_id = ?
    GROUP BY 1 ORDER BY minutes DESC LIMIT 10
""", player_id)

## Query your own DataFrames with SQL

DuckDB can query any pandas DataFrame in scope by its variable name. This runs locally; it doesn't upload anything to MotherDuck.

In [ ]:
duckdb.sql("SELECT matchup, pts FROM games WHERE pts >= 30 ORDER BY pts DESC").df()